<a href="https://colab.research.google.com/github/look4pritam/NetflixPrize/blob/master/Notebooks/PoC/Alternating_Least_Squares.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Netflix Prize

## Install Python packages.

In [ ]:
!pip install numpy pandas scipy tqdm

## Set the project root directory.

In [ ]:
import os

root_dir = '/content/'
os.chdir(root_dir)

!ls -al

total 20
drwxr-xr-x 1 root root 4096 Apr 20 06:38 .
drwxr-xr-x 1 root root 4096 Apr 20 06:26 ..
drwxr-xr-x 4 root root 4096 Apr 16 13:28 .config
-rw-r--r-- 1 root root   67 Apr 20 06:38 kaggle.json
drwxr-xr-x 1 root root 4096 Apr 16 13:28 sample_data


## Download the Netflix Prize dataset from Kaggle.

### Set the Kaggle API key.

### Generate and upload 'kaggle.json' file.

In [ ]:
os.environ['KAGGLE_CONFIG_DIR'] = root_dir
!chmod 600 /content/kaggle.json

### Download the Netflix Prize dataset.

In [ ]:
!kaggle datasets download netflix-inc/netflix-prize-data
!ls -al

Dataset URL: https://www.kaggle.com/datasets/netflix-inc/netflix-prize-data
License(s): other
100% 683M/683M [00:43<00:00, 16.4MB/s]

total 699436
drwxr-xr-x 1 root root      4096 Apr 20 06:38 .
drwxr-xr-x 1 root root      4096 Apr 20 06:26 ..
drwxr-xr-x 4 root root      4096 Apr 16 13:28 .config
-rw------- 1 root root        67 Apr 20 06:38 kaggle.json
-rw-r--r-- 1 root root 716193814 Nov 13  2019 netflix-prize-data.zip
drwxr-xr-x 1 root root      4096 Apr 16 13:28 sample_data


In [ ]:
import os
import zipfile

### Extract the Netflix Prize dataset.

In [ ]:
zip_path = "netflix-prize-data.zip"
dataset_path = "netflix_dataset"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(dataset_path)

print("Extracted Netflix Prize dataset from:", os.listdir(dataset_path))

Extracted Netflix Prize dataset from: ['probe.txt', 'combined_data_3.txt', 'README', 'combined_data_2.txt', 'movie_titles.csv', 'combined_data_4.txt', 'qualifying.txt', 'combined_data_1.txt']


## Load the Netflix Prize dataset.

### Import Python packages.

In [ ]:
import numpy as np
import pandas as pd
from tqdm import tqdm

In [ ]:
def load_netflix_data(filepath, max_rows=500000):

    user_ids = []
    movie_ids = []
    ratings = []

    movie_id = None
    count = 0

    with open(filepath, "r") as f:

        for line in f:

            if ":" in line:
                movie_id = int(line.replace(":", "").strip())

            else:
                user, rating, _ = line.split(",")

                user_ids.append(int(user))
                movie_ids.append(movie_id)
                ratings.append(float(rating))

                count += 1

                if count >= max_rows:
                    break

    df = pd.DataFrame({
        "user": user_ids,
        "movie": movie_ids,
        "rating": ratings
    })

    return df

In [ ]:
file_path = "netflix_dataset/combined_data_1.txt"
df = load_netflix_data(file_path, max_rows=500000)
df.head()

,user,movie,rating
0,1488844,1,3.0
1,822109,1,5.0
2,885013,1,4.0
3,30878,1,4.0
4,823519,1,3.0


In [ ]:
user_map = {user:index for index,user in enumerate(df.user.unique())}
movie_map = {movie:index for index,movie in enumerate(df.movie.unique())}

df["user_idx"] = df.user.map(user_map)
df["movie_idx"] = df.movie.map(movie_map)

user_count = len(user_map)
movie_count = len(movie_map)

print("Number of users: ", user_count)
print("Number of movies: ", movie_count)

Number of users:  215008
Number of movies:  148


## Compute the sparse matrix in COO format.

### COO format:
- Stores only non-zero entries
- Efficient for large sparse data

### Components:
- matrix.row - user indices
- matrix.col - movie indices
- matrix.data - ratings

In [ ]:
from scipy.sparse import coo_matrix

In [ ]:
rating_matrix = coo_matrix(
    (df.rating, (df.user_idx, df.movie_idx)),
    shape=(user_count, movie_count)
)

## Factorize matrix using Alternating Least Squares (ALS).

In [ ]:
def matrix_factorization(rating_matrix, embedding_size=20, iterations=5, regularization_factor=0.1):

    user_count, sample_count = rating_matrix.shape

    user_features = np.random.normal(scale=1./embedding_size, size=(user_count, embedding_size))
    movie_features = np.random.normal(scale=1./embedding_size, size=(sample_count, embedding_size))

    rows = rating_matrix.row
    cols = rating_matrix.col
    ratings = rating_matrix.data

    for current_iteration in range(iterations):

        print("Iteration: ", current_iteration+1)

        # Update Users
        for u in tqdm(range(user_count)):

            idx = rows == u
            items = cols[idx]
            r = ratings[idx]

            if len(items) == 0:
                continue

            V_i = movie_features[items]

            A = V_i.T @ V_i + regularization_factor*np.eye(embedding_size)
            b = V_i.T @ r

            user_features[u] = np.linalg.solve(A, b)

        # Update Movies
        for i in tqdm(range(sample_count)):

            idx = cols == i
            users = rows[idx]
            r = ratings[idx]

            if len(users) == 0:
                continue

            U_u = user_features[users]

            A = U_u.T @ U_u + regularization_factor*np.eye(embedding_size)
            b = U_u.T @ r

            movie_features[i] = np.linalg.solve(A, b)

    return user_features, movie_features

In [ ]:
user_features, movie_features = matrix_factorization(rating_matrix, embedding_size=20, iterations=5)

Iteration:  1


100%|██████████| 148/148 [00:00<00:00, 571.59it/s]


Iteration:  2


100%|██████████| 148/148 [00:00<00:00, 592.56it/s]


Iteration:  3


100%|██████████| 148/148 [00:00<00:00, 200.38it/s]


Iteration:  4


100%|██████████| 148/148 [00:00<00:00, 601.55it/s]


Iteration:  5


100%|██████████| 148/148 [00:00<00:00, 546.67it/s]


## Compute the predicted matrix.

In [ ]:
predicted_matrix = user_features @ movie_features.T

## Computer the RMSE metric.

In [ ]:
def compute_rmse(rating_matrix, predicted_matrix):

    rows = rating_matrix.row
    cols = rating_matrix.col
    ground_truth_ratings = rating_matrix.data

    predicted_ratings = predicted_matrix[rows, cols]

    rmse = np.sqrt(np.mean((ground_truth_ratings - predicted_ratings)**2))

    return rmse

In [ ]:
rmse = compute_rmse(rating_matrix, predicted_matrix)
print("RMSE: ", rmse)

RMSE:  0.09664026449553208
